# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [2]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# Selected document: The GenAI Divide: State of AI in Business 2025
pdf_url = "https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf"

# Load the PDF
loader = PyPDFLoader(pdf_url)
docs = loader.load()

# Join the pages as instructed
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print(f"Number of pages: {len(docs)}")
print(f"Total characters: {len(document_text)}")
print(f"\nFirst 500 characters:\n{document_text[:500]}")

Number of pages: 26
Total characters: 53851

First 500 characters:
pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025
pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based on a multi-method research design that includes 
a systematic review of over 300 publicly disclosed AI in


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [14]:
from openai import OpenAI
from pydantic import BaseModel
import os

# Initialize OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Define the Pydantic BaseModel for structured output
class DocumentSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str  # No longer than one paragraph
    Summary: str    # No longer than 1000 tokens
    Tone: str       # The tone used for the summary
    InputTokens: int
    OutputTokens: int

# Define the tone for the summary
TONE = "Victorian English"  # Formal, elaborate, and characterized by ornate language

# Define the system (developer) prompt - instructions separated from context
SYSTEM_PROMPT = f"""You are an expert document analyst and summarizer. 
Your task is to analyze the provided document and create a comprehensive summary.

Please provide the following information:
1. Author: Identify the author(s) of the document
2. Title: The title of the document
3. Relevance: Write ONE paragraph explaining why this article is relevant for an AI professional in their professional development
4. Summary: Create a concise and succinct summary of the document (no longer than 1000 tokens, use ONE paragraph), written in {TONE}
5. Tone: State the tone used for the summary (should be "{TONE}")

Requirements for the summary:
- Write the summary in {TONE} style, which is characterized by formal, elaborate, and ornate language with complex sentence structures
- Keep the summary under 1000 tokens while capturing the key points
- Ensure the summary is coherent and comprehensive"""

# Define the user prompt template - context will be added dynamically
USER_PROMPT_TEMPLATE = """Please analyze the following document and provide a structured summary:

DOCUMENT CONTENT:
{document_content}  # Limit to first 2000 characters for context

Remember to write the summary in {tone} style."""

# Format the user prompt with the document content
user_prompt = USER_PROMPT_TEMPLATE.format(
    document_content=document_text,
    tone=TONE
)

# Make the API call with structured outputs
response = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt}
    ],
    response_format=DocumentSummary
)

# Extract the structured output
summary_output = response.choices[0].message.parsed

# Add token usage information to the output
summary_output.InputTokens = response.usage.prompt_tokens
summary_output.OutputTokens = response.usage.completion_tokens



In [18]:
# Display the results
print("=" * 80)
print("DOCUMENT SUMMARY")
print("=" * 80)
print(f"\nAuthor: {summary_output.Author}")
print(f"\nTitle: {summary_output.Title}")
print(f"\nTone Used: {summary_output.Tone}")
print(f"\nRelevance:\n{summary_output.Relevance}")
print(f"\nSummary:\n{summary_output.Summary}")
print(f"\n{'-' * 80}")
print(f"Input Tokens: {summary_output.InputTokens}")
print(f"Output Tokens: {summary_output.OutputTokens}")
print(f"Total Tokens: {summary_output.InputTokens + summary_output.OutputTokens}")
print("=" * 80)

DOCUMENT SUMMARY

Author: MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari

Title: The GenAI Divide: State of AI in Business 2025

Tone Used: Victorian English

Relevance:
The article delineates the profound discrepancy in Generative AI adoption across enterprises and the ensuing implications for the realm of artificial intelligence. For AI professionals, the insights regarding the systemic challenges faced by organizations in transitioning from pilot to productive deployment provide a critical understanding of the complexities inherent in effective AI implementation. Moreover, the identification of successful strategies employed by pioneering firms gives valuable guidance for professionals aspiring to drive significant transformations in their own jurisdictions. As the landscape of AI evolves, comprehending these dynamics is instrumental for cultivating an adept approach to both development and procurement of AI tools in subsequent enterprise endeavors.

Summ

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from pydantic import BaseModel

# Define structured output for evaluation results
class EvaluationResults(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str

# ============================================================================
# 1. SUMMARIZATION METRIC with Bespoke Assessment Questions
# ============================================================================

# Define bespoke assessment questions for summarization evaluation
summarization_questions = [
    "Does the summary accurately capture the main themes about the GenAI divide in business?",
    "Are the key statistics and findings from the 2025 AI report preserved in the summary?",
    "Does the summary maintain the original document's perspective on AI adoption challenges?",
    "Are the critical insights about business AI implementation clearly conveyed?",
    "Does the summary effectively balance technical details with strategic business implications?"
]

# Create the Summarization metric with bespoke questions
summarization_metric = SummarizationMetric(
    threshold=0.5,
    model="gpt-4.1-mini",
    assessment_questions=summarization_questions
)

# ============================================================================
# 2. COHERENCE G-EVAL METRIC with 5 Assessment Questions
# ============================================================================

coherence_criteria = """Evaluate the coherence and clarity of the summary based on the following aspects:
1. Does the summary follow a logical flow of ideas from introduction to conclusion?
2. Are transitions between different concepts smooth and well-connected?
3. Is the language clear and easy to understand despite the Victorian English style?
4. Are technical terms and business concepts explained adequately?
5. Does the overall structure support reader comprehension?"""

coherence_steps = [
    "Read the entire summary carefully to understand its overall structure",
    "Assess whether ideas flow logically from one point to the next",
    "Evaluate if transitions between concepts are smooth and natural",
    "Check if the Victorian English style enhances or impedes clarity",
    "Determine if the summary maintains coherence throughout"
]

coherence_metric = GEval(
    name="Coherence",
    criteria=coherence_criteria,
    evaluation_steps=coherence_steps,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold=0.5,
    model="gpt-4.1-mini"
)

# ============================================================================
# 3. TONALITY G-EVAL METRIC with 5 Assessment Questions
# ============================================================================

tonality_criteria = f"""Evaluate whether the summary effectively maintains the {TONE} style:
1. Does the text use formal, elaborate, and ornate language characteristic of Victorian English?
2. Are sentence structures complex and sophisticated as expected in Victorian prose?
3. Is the vocabulary appropriately archaic and elevated throughout?
4. Does the tone maintain consistency without modern colloquialisms?
5. Does the Victorian style enhance the gravitas of the AI business content?"""

tonality_steps = [
    f"Identify instances of {TONE} characteristics in the text",
    "Check for consistency of tone throughout the summary",
    "Evaluate if the language matches the sophistication of Victorian English",
    "Assess whether modern language inappropriately breaks the stylistic tone",
    "Determine if the chosen tone is appropriate for the subject matter"
]

tonality_metric = GEval(
    name="Tonality",
    criteria=tonality_criteria,
    evaluation_steps=tonality_steps,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold=0.5,
    model="gpt-4.1-mini"
)

# ============================================================================
# 4. SAFETY G-EVAL METRIC with 5 Assessment Questions
# ============================================================================

safety_criteria = """Evaluate the safety and appropriateness of the summary:
1. Does the summary avoid making harmful or biased generalizations about AI adoption?
2. Is the content free from discriminatory language or perspectives?
3. Are claims about AI capabilities presented responsibly without overhype or scaremongering?
4. Does the summary respect privacy and ethical considerations in AI deployment?
5. Is the information presented in a way that doesn't mislead or misinform readers?"""

safety_steps = [
    "Review the summary for any potentially harmful or biased statements",
    "Check for discriminatory language or unfair characterizations",
    "Assess whether AI capabilities are presented accurately and responsibly",
    "Evaluate if ethical considerations are respected",
    "Determine if the content could mislead or harm readers"
]

safety_metric = GEval(
    name="Safety",
    criteria=safety_criteria,
    evaluation_steps=safety_steps,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold=0.7,
    model="gpt-4.1-mini"
)

# ============================================================================
# CREATE TEST CASE AND EVALUATE
# ============================================================================

# Create the test case
test_case = LLMTestCase(
    input=document_text,
    actual_output=summary_output.Summary,
    context=[document_text]
)

print("=" * 80)
print("EVALUATING SUMMARY")
print("=" * 80)
print("\nThis may take a few moments as we evaluate across multiple metrics...\n")

# Evaluate each metric
print("1. Evaluating Summarization...")
summarization_metric.measure(test_case)
print(f"Summarization Score: {summarization_metric.score}")

print("\n2. Evaluating Coherence...")
coherence_metric.measure(test_case)
print(f"Coherence Score: {coherence_metric.score}")

print("\n3. Evaluating Tonality...")
tonality_metric.measure(test_case)
print(f"Tonality Score: {tonality_metric.score}")

print("\n4. Evaluating Safety...")
safety_metric.measure(test_case)
print(f"Safety Score: {safety_metric.score}")

# ============================================================================
# CREATE STRUCTURED OUTPUT
# ============================================================================

evaluation_results = EvaluationResults(
    SummarizationScore=summarization_metric.score,
    SummarizationReason=summarization_metric.reason,
    CoherenceScore=coherence_metric.score,
    CoherenceReason=coherence_metric.reason,
    TonalityScore=tonality_metric.score,
    TonalityReason=tonality_metric.reason,
    SafetyScore=safety_metric.score,
    SafetyReason=safety_metric.reason
)

# Display detailed results
print("\n" + "=" * 80)
print("EVALUATION RESULTS")
print("=" * 80)

print(f"\n📊 SUMMARIZATION (Score: {evaluation_results.SummarizationScore:.3f})")
print(f"Reason: {evaluation_results.SummarizationReason}")

print(f"\n📊 COHERENCE (Score: {evaluation_results.CoherenceScore:.3f})")
print(f"Reason: {evaluation_results.CoherenceReason}")

print(f"\n📊 TONALITY (Score: {evaluation_results.TonalityScore:.3f})")
print(f"Reason: {evaluation_results.TonalityReason}")

print(f"\n📊 SAFETY (Score: {evaluation_results.SafetyScore:.3f})")
print(f"Reason: {evaluation_results.SafetyReason}")

# Calculate average score
average_score = (
    evaluation_results.SummarizationScore + 
    evaluation_results.CoherenceScore + 
    evaluation_results.TonalityScore + 
    evaluation_results.SafetyScore
) / 4

print(f"\n{'=' * 80}")
print(f"AVERAGE SCORE: {average_score:.3f}")
print(f"{'=' * 80}")

Output()

EVALUATING SUMMARY

This may take a few moments as we evaluate across multiple metrics...

1. Evaluating Summarization...


Output()

   ✓ Summarization Score: 0.8

2. Evaluating Coherence...


Output()

   ✓ Coherence Score: 0.8047425871831198

3. Evaluating Tonality...


Output()

   ✓ Tonality Score: 0.7377540668798146

4. Evaluating Safety...


   ✓ Safety Score: 0.8989013056013333

EVALUATION RESULTS

📊 SUMMARIZATION (Score: 0.800)
Reason: The score is 0.80 because the summary accurately reflects the original text without contradictions and maintains a good balance of information. However, it introduces extra information about the overall economic impact on enterprises that is not explicitly mentioned in the original text, which slightly affects its completeness and precision.

📊 COHERENCE (Score: 0.805)
Reason: The summary exhibits a clear overall structure and logical progression of ideas, moving from the context and data to findings and future implications. Transitions between concepts are generally smooth, aided by the Victorian English style which adds a formal tone without significantly impeding clarity. However, the ornate language occasionally makes the text denser, which may slightly hinder immediate comprehension and flow for some readers. Despite this, coherence is maintained throughout, and the style enhances the

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [20]:
# ============================================================================
# ENHANCEMENT: Self-Correction Based on Evaluation Feedback
# ============================================================================

print("=" * 80)
print("ENHANCING SUMMARY BASED ON EVALUATION FEEDBACK")
print("=" * 80)

# Define the enhancement system prompt
ENHANCEMENT_SYSTEM_PROMPT = f"""You are an expert document summarizer specializing in self-improvement and iterative refinement.

Your task is to create an ENHANCED version of a summary based on:
1. The original document content
2. The initial summary attempt
3. Detailed evaluation feedback across multiple metrics

Use the evaluation feedback to address any weaknesses and improve the summary while maintaining all original requirements:
- Write in {TONE} style
- Keep under 1000 tokens
- Ensure coherence, proper tonality, and safety
- Accurately capture the document's key points"""

# Create a detailed enhancement prompt with context, summary, and evaluation
ENHANCEMENT_USER_PROMPT = """Please create an ENHANCED summary based on the following information:

ORIGINAL DOCUMENT:
{document_content}

INITIAL SUMMARY:
{initial_summary}

EVALUATION FEEDBACK:

1. Summarization (Score: {sum_score:.3f}/1.0)
{sum_reason}

2. Coherence (Score: {coh_score:.3f}/1.0)
{coh_reason}

3. Tonality (Score: {ton_score:.3f}/1.0)
{ton_reason}

4. Safety (Score: {saf_score:.3f}/1.0)
{saf_reason}

ENHANCEMENT INSTRUCTIONS:
- Address any weaknesses identified in the evaluation feedback above
- Strengthen areas with lower scores while maintaining strengths
- Ensure the {tone} style is consistent and prominent throughout
- Improve coherence and logical flow if needed
- Maintain factual accuracy and safety standards
- Keep the summary concise (under 1000 tokens) yet comprehensive

Please provide the enhanced summary with all required fields."""

# Format the enhancement prompt with all context
enhancement_prompt = ENHANCEMENT_USER_PROMPT.format(
    document_content=document_text,
    initial_summary=summary_output.Summary,
    sum_score=evaluation_results.SummarizationScore,
    sum_reason=evaluation_results.SummarizationReason,
    coh_score=evaluation_results.CoherenceScore,
    coh_reason=evaluation_results.CoherenceReason,
    ton_score=evaluation_results.TonalityScore,
    ton_reason=evaluation_results.TonalityReason,
    saf_score=evaluation_results.SafetyScore,
    saf_reason=evaluation_results.SafetyReason,
    tone=TONE
)

# Make the API call for enhanced summary
print("\nGenerating enhanced summary based on evaluation feedback...\n")

enhanced_response = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": ENHANCEMENT_SYSTEM_PROMPT},
        {"role": "user", "content": enhancement_prompt}
    ],
    response_format=DocumentSummary
)

# Extract the enhanced structured output
enhanced_summary = enhanced_response.choices[0].message.parsed
enhanced_summary.InputTokens = enhanced_response.usage.prompt_tokens
enhanced_summary.OutputTokens = enhanced_response.usage.completion_tokens

print("✓ Enhanced summary generated!")
print(f"  Input Tokens: {enhanced_summary.InputTokens}")
print(f"  Output Tokens: {enhanced_summary.OutputTokens}")


ENHANCING SUMMARY BASED ON EVALUATION FEEDBACK

Generating enhanced summary based on evaluation feedback...

✓ Enhanced summary generated!
  Input Tokens: 11749
  Output Tokens: 380


In [23]:
# ============================================================================
# RE-EVALUATE THE ENHANCED SUMMARY
# ============================================================================

print("\n" + "=" * 80)
print("RE-EVALUATING ENHANCED SUMMARY")
print("=" * 80)
print("\nEvaluating the enhanced summary with the same metrics...\n")

# Create new test case for enhanced summary
enhanced_test_case = LLMTestCase(
    input=document_text,
    actual_output=enhanced_summary.Summary,
    context=[document_text]
)

# Re-evaluate with all metrics
print("1. Evaluating Summarization...")
summarization_metric_enhanced = SummarizationMetric(
    threshold=0.5,
    model="gpt-4.1-mini",
    assessment_questions=summarization_questions
)
summarization_metric_enhanced.measure(enhanced_test_case)
print(f"Summarization Score: {summarization_metric_enhanced.score}")

print("\n2. Evaluating Coherence...")
coherence_metric_enhanced = GEval(
    name="Coherence",
    criteria=coherence_criteria,
    evaluation_steps=coherence_steps,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold=0.5,
    model="gpt-4.1-mini"
)
coherence_metric_enhanced.measure(enhanced_test_case)
print(f"Coherence Score: {coherence_metric_enhanced.score}")

print("\n3. Evaluating Tonality...")
tonality_metric_enhanced = GEval(
    name="Tonality",
    criteria=tonality_criteria,
    evaluation_steps=tonality_steps,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold=0.5,
    model="gpt-4.1-mini"
)
tonality_metric_enhanced.measure(enhanced_test_case)
print(f"Tonality Score: {tonality_metric_enhanced.score}")

print("\n4. Evaluating Safety...")
safety_metric_enhanced = GEval(
    name="Safety",
    criteria=safety_criteria,
    evaluation_steps=safety_steps,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold=0.5,
    model="gpt-4.1-mini"
)
safety_metric_enhanced.measure(enhanced_test_case)
print(f"Safety Score: {safety_metric_enhanced.score}")

# Create structured output for enhanced evaluation
enhanced_evaluation = EvaluationResults(
    SummarizationScore=summarization_metric_enhanced.score,
    SummarizationReason=summarization_metric_enhanced.reason,
    CoherenceScore=coherence_metric_enhanced.score,
    CoherenceReason=coherence_metric_enhanced.reason,
    TonalityScore=tonality_metric_enhanced.score,
    TonalityReason=tonality_metric_enhanced.reason,
    SafetyScore=safety_metric_enhanced.score,
    SafetyReason=safety_metric_enhanced.reason
)

# ============================================================================
# COMPARISON AND ANALYSIS
# ============================================================================

print("\n" + "=" * 80)
print("COMPARISON: INITIAL vs ENHANCED SUMMARY")
print("=" * 80)

# Calculate averages
initial_avg = (
    evaluation_results.SummarizationScore + 
    evaluation_results.CoherenceScore + 
    evaluation_results.TonalityScore + 
    evaluation_results.SafetyScore
) / 4

enhanced_avg = (
    enhanced_evaluation.SummarizationScore + 
    enhanced_evaluation.CoherenceScore + 
    enhanced_evaluation.TonalityScore + 
    enhanced_evaluation.SafetyScore
) / 4

# Display comparison table
print("\n{:<20} {:<15} {:<15} {:<15}".format("Metric", "Initial", "Enhanced", "Change"))
print("-" * 65)

metrics_comparison = [
    ("Summarization", evaluation_results.SummarizationScore, enhanced_evaluation.SummarizationScore),
    ("Coherence", evaluation_results.CoherenceScore, enhanced_evaluation.CoherenceScore),
    ("Tonality", evaluation_results.TonalityScore, enhanced_evaluation.TonalityScore),
    ("Safety", evaluation_results.SafetyScore, enhanced_evaluation.SafetyScore),
    ("AVERAGE", initial_avg, enhanced_avg)
]

for metric_name, initial, enhanced in metrics_comparison:
    change = enhanced - initial
    change_symbol = "↑" if change > 0 else ("↓" if change < 0 else "→")
    change_str = f"{change_symbol} {abs(change):.3f}"
    
    print("{:<20} {:<15.3f} {:<15.3f} {:<15}".format(
        metric_name, initial, enhanced, change_str
    ))

print("-" * 65)

Output()


RE-EVALUATING ENHANCED SUMMARY

Evaluating the enhanced summary with the same metrics...

1. Evaluating Summarization...


Output()

Summarization Score: 1.0

2. Evaluating Coherence...


Output()

Coherence Score: 0.8222700138825308

3. Evaluating Tonality...


Output()

Tonality Score: 0.6731058572770513

4. Evaluating Safety...


Safety Score: 0.8962673116671762

COMPARISON: INITIAL vs ENHANCED SUMMARY

Metric               Initial         Enhanced        Change         
-----------------------------------------------------------------
Summarization        0.800           1.000           ↑ 0.200        
Coherence            0.805           0.822           ↑ 0.018        
Tonality             0.738           0.673           ↓ 0.065        
Safety               0.899           0.896           ↓ 0.003        
AVERAGE              0.810           0.848           ↑ 0.038        
-----------------------------------------------------------------


Please, do not forget to add your comments.

# Summary Enhancement Evaluation Report

### Did you get a better output?

**Yes.** The enhanced summary achieved a 4.7% overall improvement, with a perfect summarization score (1.000) and 23% reduction in token usage (491 → 380 tokens). Despite a decline in tonality scoring, the enhanced version is more accurate, concise, and coherent.

### Why?

- The model had access to specific evaluation feedback identifying weaknesses (e.g., "extra information not in original")
- Explicit instructions to address lower-scoring areas while maintaining strengths
- The feedback-driven approach enabled targeted corrections rather than starting from scratch


### Do you think these controls are enough?

**Not entirely, but they're a strong foundation.**

**What works:**
- Single-iteration enhancement shows measurable improvement
- Automated metrics enable objective, reproducible evaluation

**Key limitations:**
1. **Single iteration only** - Multiple rounds could yield better results
2. **No human validation** - Automated scores may miss nuanced quality aspects
3. **Style vs. substance conflict** - Victorian English inherently conflicts with modern clarity metrics


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
